In [1]:
from preprocessing.data_cleaning import preprocess
from config import DTYPE_MAPPING

In [2]:
from pathlib import Path
DATA_DIR = Path("../data").resolve() / "melb_jun_25"                 
LISTINGS_CSV = DATA_DIR / "listings.csv"
REVIEWS_CSV  = DATA_DIR / "reviews.csv"
OUTPUT_DIR = Path("../output").resolve()

In [3]:
import pandas as pd
df = pd.read_csv(LISTINGS_CSV)

In [34]:
processed_df, imputation_pipeline = preprocess(df, DTYPE_MAPPING)

In [10]:
len(processed_df)

18539

In [8]:
import pandas as pd
import numpy as np

def preprocess_datetime(df, date_cols, reference_date=None, fill_na=True):
    df = df.copy()
    if reference_date is None:
        reference_date = pd.Timestamp.today()
    else:
        reference_date = pd.to_datetime(reference_date)
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        if fill_na:
            median_date = df[col].median()
            df[col] = df[col].fillna(median_date)
        df[f"{col}_year"] = df[col].dt.year
        df[f"{col}_month"] = df[col].dt.month
        df[f"{col}_day"] = df[col].dt.day
        df[f"{col}_dayofweek"] = df[col].dt.dayofweek
        df[f"{col}_is_weekend"] = df[col].dt.dayofweek.isin([5,6]).astype(int)
        df[f"{col}_quarter"] = df[col].dt.quarter
        df[f"{col}_month_sin"] = np.sin(2 * np.pi * df[f"{col}_month"] / 12)
        df[f"{col}_month_cos"] = np.cos(2 * np.pi * df[f"{col}_month"] / 12)
        df[f"{col}_dow_sin"] = np.sin(2 * np.pi * df[f"{col}_dayofweek"] / 7)
        df[f"{col}_dow_cos"] = np.cos(2 * np.pi * df[f"{col}_dayofweek"] / 7)
        df[f"{col}_days_since"] = (reference_date - df[col]).dt.days
    return df

In [72]:
processed_df = preprocess_datetime(processed_df, date_cols=DTYPE_MAPPING['date'])

In [64]:
processed_df['neighbourhood'].value_counts()

neighbourhood
Neighborhood highlights    18539
Name: count, dtype: int64

In [68]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import re

def categorical_encoding_pipeline(mapping):
    return (
        ColumnTransformer(
            [
                ("onehot", OneHotEncoder(sparse_output=False), mapping["nominal"]),
                #("multihot", CountVectorizer(binary=True), ["amenities", "host_verifications"])
            ],
            remainder='passthrough'
        ),
        mapping["nominal"]
    )

def categorical_encoding_fit_transform(preprocessor, df, target_cols):
    result_array = preprocessor.fit_transform(df)
    
    columns = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "onehot":
            columns.extend(preprocessor.named_transformers_['onehot'].get_feature_names_out(cols))
        elif name == "remainder":
            passthrough_cols = [c for c in df.columns if c not in target_cols]
            columns.extend(passthrough_cols)
    
    return pd.DataFrame(result_array, columns=columns, index=df.index)

In [69]:
preprocessor, target_cols = categorical_encoding_pipeline(DTYPE_MAPPING)
preprocessor

,transformers,"[('onehot', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False


In [70]:
processed_df = categorical_encoding_fit_transform(preprocessor, processed_df, target_cols)

In [71]:
processed_df

,host_name_(Maria) ANNA,host_name_A&E,host_name_ACD Apartments,host_name_Aaditya,host_name_Aaron,host_name_Aarush,host_name_Aayushi,host_name_Ab,host_name_Abbie,host_name_Abby,...,last_review_month,last_review_day,last_review_dayofweek,last_review_is_weekend,last_review_quarter,last_review_month_sin,last_review_month_cos,last_review_dow_sin,last_review_dow_cos,last_review_days_since
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,23,2,0,2,0.866025,-0.5,0.974928,-0.222521,174
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6,4,2,0,2,0.0,-1.0,0.974928,-0.222521,132
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,6,6,1,2,0.866025,-0.5,-0.781831,0.62349,191
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,8,2,0,1,0.5,0.866025,0.974928,-0.222521,279
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5,24,5,1,2,0.5,-0.866025,-0.974928,-0.222521,143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25796,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,28,0,0,2,0.866025,-0.5,0.0,1.0,169
25797,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,28,0,0,2,0.866025,-0.5,0.0,1.0,169
25798,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,28,0,0,2,0.866025,-0.5,0.0,1.0,169
25799,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,28,0,0,2,0.866025,-0.5,0.0,1.0,169


In [73]:
def get_percent_nan(df, target):
    return df[target].isnull().sum() / len(df)

In [78]:
processed_df['amenities']

0        [Room-darkening shades, Kitchen, BBQ grill: ga...
3        [Kitchen, Stainless steel single oven, Private...
4        [Room-darkening shades, Kitchen, Fire extingui...
5        [AC - split type ductless system, Kitchen, Sto...
6        [AC - split type ductless system, Kitchen, Win...
                               ...                        
25796    [Kitchen, Dedicated workspace, Beach access – ...
25797    [AC - split type ductless system, Kitchen, Pri...
25798    [Kitchen, Dedicated workspace, Lock on bedroom...
25799    [Kitchen, Dedicated workspace, Lake access, Wa...
25800    [Room-darkening shades, Kitchen, Stove, Clothi...
Name: amenities, Length: 18539, dtype: object

In [77]:
for c in processed_df.columns:
    if (n := get_percent_nan(processed_df,c)) > 0:
        print(c)
        print(n)

description
0.013161443443551432
neighborhood_overview
0.547710232482874
host_about
0.4287717784130751
